# Loss Functions

Implement MSE, MAE, binary cross-entropy, and categorical cross-entropy from scratch in PyTorch, validate each against `torch.nn.functional`, and learn when to use which loss.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()` — never hardcoded. On Apple Silicon this runs on mps.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)


running on: mps


## Synthetic data

We create small regression and binary-classification tensors on the configured device. Every from-scratch implementation is validated against `torch.nn.functional`.

In [2]:
import torch.nn.functional as F

torch.manual_seed(42)
N = 64

# Regression targets in [-3, 3]
y_reg = torch.randn(N, device=device)
y_hat = y_reg + 0.5 * torch.randn(N, device=device)  # noisy predictions

# Binary classification: targets in {0, 1}, logits unbounded
y_bin = (torch.rand(N, device=device) > 0.5).float()
logits_bin = torch.randn(N, device=device)

# Multiclass: 4 classes
C = 4
y_mc = torch.randint(0, C, (N,), device=device)
logits_mc = torch.randn(N, C, device=device)

print("regression targets:", tuple(y_reg.shape), "on", y_reg.device.type)
print("binary targets:", tuple(y_bin.shape), "| logits:", tuple(logits_bin.shape))
print("multiclass targets:", tuple(y_mc.shape), "| logits:", tuple(logits_mc.shape))


regression targets: (64,) on mps
binary targets: (64,) | logits: (64,)
multiclass targets: (64,) | logits: (64, 4)


## Mean Squared Error (MSE)

MSE corresponds to maximum likelihood under a Gaussian residual model with constant variance. Large residuals receive quadratically larger gradients, making MSE sensitive to outliers.

$$\text{MSE}(\hat{y}, y) = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2$$

Gradient with respect to \(\hat{y}_i\): \(\frac{2}{N}(\hat{y}_i - y_i)\).

In [3]:
def mse_loss(y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Mean squared error: (1/N) sum_i (yhat_i - y_i)^2."""
    return ((y_hat - y) ** 2).mean()


scratch = mse_loss(y_hat, y_reg)
ref = F.mse_loss(y_hat, y_reg)
print(f"MSE from scratch: {scratch.item():.6f}")
print(f"F.mse_loss:       {ref.item():.6f}")
assert torch.allclose(scratch, ref, atol=1e-6), f"MSE mismatch: {scratch} vs {ref}"
print("MSE matches torch ✓")


MSE from scratch: 0.185621
F.mse_loss:       0.185621
MSE matches torch ✓


## Mean Absolute Error (MAE)

MAE corresponds to maximum likelihood under a Laplace residual model. Its subgradient is \(\text{sign}(\hat{y}_i - y_i)\), so outliers contribute a bounded gradient of ±1 regardless of magnitude — making MAE more robust than MSE.

$$\text{MAE}(\hat{y}, y) = \frac{1}{N}\sum_{i=1}^{N}|\hat{y}_i - y_i|$$

Unlike MSE, MAE is not differentiable at zero, which can cause issues with gradient-based optimizers.

In [4]:
def mae_loss(y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Mean absolute error: (1/N) sum_i |yhat_i - y_i|."""
    return (y_hat - y).abs().mean()


scratch = mae_loss(y_hat, y_reg)
ref = F.l1_loss(y_hat, y_reg)
print(f"MAE from scratch: {scratch.item():.6f}")
print(f"F.l1_loss:        {ref.item():.6f}")
assert torch.allclose(scratch, ref, atol=1e-6), f"MAE mismatch: {scratch} vs {ref}"
print("MAE matches torch ✓")


MAE from scratch: 0.325836
F.l1_loss:        0.325836
MAE matches torch ✓


### MSE vs MAE: outlier sensitivity

MSE penalizes large residuals quadratically; MAE penalizes them linearly. This plot shows how the per-sample loss contribution grows with the residual magnitude.

In [5]:
r = torch.linspace(-3, 3, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r.numpy(), (r ** 2).numpy(), label="MSE (squared)")
ax.plot(r.numpy(), r.abs().numpy(), label="MAE (absolute)")
ax.set_xlabel("residual")
ax.set_ylabel("per-sample loss")
ax.set_title("MSE vs MAE penalty shape")
ax.legend()
plt.tight_layout()
plt.show()


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_17130/463253803.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Binary Cross-Entropy (BCE)

BCE is the negative log-likelihood of a Bernoulli distribution. For predicted probability \(p = \sigma(z)\) and target \(y \in \{0, 1\}\):

$$\text{BCE}(p, y) = -y \log p - (1 - y) \log(1 - p)$$

**Important distinction:** BCE operates on *probabilities* \(p \in (0, 1)\). BCE-with-logits operates on raw *logits* \(z \in \mathbb{R}\) and fuses the sigmoid into a numerically stable log-sum-exp form, avoiding floating-point underflow near 0 and 1.

In [6]:
def bce_loss(probs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Binary cross-entropy from probabilities (not logits).

    Clamps probabilities to avoid log(0).
    """
    eps = torch.finfo(probs.dtype).eps
    p = probs.clamp(eps, 1.0 - eps)
    return -(targets * p.log() + (1 - targets) * (1 - p).log()).mean()


def bce_with_logits_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Numerically stable BCE-with-logits using the identity:

        log(1 + exp(z)) = max(z, 0) + log(1 + exp(-|z|))

    This avoids computing sigma(z) explicitly, which saturates for large |z|.
    """
    # log(sigma(z)) = z - softplus(z) = z - max(z,0) - log(1+exp(-|z|))
    # Numerically stable form for F.binary_cross_entropy_with_logits:
    return (torch.relu(logits) - logits * targets + (1 + (-logits.abs()).exp()).log()).mean()


probs = torch.sigmoid(logits_bin)

scratch_bce = bce_loss(probs, y_bin)
ref_bce = F.binary_cross_entropy(probs, y_bin)
print(f"BCE (probs) scratch: {scratch_bce.item():.6f}")
print(f"F.binary_cross_entropy: {ref_bce.item():.6f}")
assert torch.allclose(scratch_bce, ref_bce, atol=1e-5), f"BCE mismatch: {scratch_bce} vs {ref_bce}"
print("BCE matches torch ✓")

scratch_bce_logits = bce_with_logits_loss(logits_bin, y_bin)
ref_bce_logits = F.binary_cross_entropy_with_logits(logits_bin, y_bin)
print(f"BCE-with-logits scratch: {scratch_bce_logits.item():.6f}")
print(f"F.binary_cross_entropy_with_logits: {ref_bce_logits.item():.6f}")
assert torch.allclose(scratch_bce_logits, ref_bce_logits, atol=1e-5), (
    f"BCE-with-logits mismatch: {scratch_bce_logits} vs {ref_bce_logits}"
)
print("BCE-with-logits matches torch ✓")


BCE (probs) scratch: 0.832699


F.binary_cross_entropy: 0.832699
BCE matches torch ✓
BCE-with-logits scratch: 0.832699
F.binary_cross_entropy_with_logits: 0.832699
BCE-with-logits matches torch ✓


### Why BCE-with-logits is numerically preferable

When logits are very large in magnitude, `sigmoid(z)` saturates to 0 or 1 in float32, so `log(p)` or `log(1-p)` becomes `-inf`. BCE-with-logits avoids computing `sigmoid` explicitly and instead uses the numerically stable log-sum-exp form.

In [7]:
# Extreme logits expose the numerical instability of sigmoid+BCE
extreme_logits = torch.tensor([100.0, -100.0, 50.0, -50.0], device=device)
extreme_targets = torch.tensor([1.0, 0.0, 1.0, 0.0], device=device)

extreme_probs = torch.sigmoid(extreme_logits)
print("sigmoid(extreme logits):", extreme_probs)
# When prob saturates to 1.0, log(1 - p) = log(0) = -inf
naive_result = bce_loss(extreme_probs, extreme_targets)
stable_result = bce_with_logits_loss(extreme_logits, extreme_targets)
ref_stable = F.binary_cross_entropy_with_logits(extreme_logits, extreme_targets)
print(f"Naive BCE (via sigmoid): {naive_result.item()}")
print(f"BCE-with-logits scratch: {stable_result.item():.6f}")
print(f"BCE-with-logits torch:   {ref_stable.item():.6f}")
# Note: naive result may be nan/inf; the logits-based form should match torch exactly
assert torch.allclose(stable_result, ref_stable, atol=1e-5)
print("Numerically stable BCE matches torch on extreme logits ✓")


sigmoid(extreme logits): 

tensor([1.0000e+00, 0.0000e+00, 1.0000e+00, 1.9287e-22], device='mps:0')


Naive BCE (via sigmoid): 1.1920928955078125e-07
BCE-with-logits scratch: 0.000000
BCE-with-logits torch:   0.000000
Numerically stable BCE matches torch on extreme logits ✓


## Categorical Cross-Entropy

For mutually exclusive classes, softmax turns logit vector \(z\) into a categorical distribution \(p_k = e^{z_k} / \sum_j e^{z_j}\). The loss is the negative log-probability of the true class:

$$\text{CE}(z, y) = -\log p_y = -z_y + \log \sum_k e^{z_k}$$

The log-sum-exp must be computed in a numerically stable way by subtracting the max logit:

$$\log \sum_k e^{z_k} = \max_k z_k + \log \sum_k e^{z_k - \max_k z_k}$$

The gradient of CE-with-softmax with respect to the logit \(z_k\) is elegantly \(p_k - \mathbb{1}[k = y]\), i.e., the difference between the predicted and true distribution.

In [8]:
def categorical_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Numerically stable categorical cross-entropy.

    Args:
        logits: shape (N, C) — raw unnormalized scores.
        targets: shape (N,) — integer class indices in [0, C).

    Returns:
        Scalar mean cross-entropy loss.
    """
    # Stable log-sum-exp: subtract per-sample max to prevent overflow
    z_max = logits.max(dim=1, keepdim=True).values  # (N, 1)
    log_sum_exp = (logits - z_max).exp().sum(dim=1).log() + z_max.squeeze(1)  # (N,)
    # Gather the true-class logit for each sample
    true_logit = logits[torch.arange(len(targets), device=logits.device), targets]  # (N,)
    return (log_sum_exp - true_logit).mean()


scratch_ce = categorical_cross_entropy(logits_mc, y_mc)
ref_ce = F.cross_entropy(logits_mc, y_mc)
print(f"Categorical CE scratch: {scratch_ce.item():.6f}")
print(f"F.cross_entropy:        {ref_ce.item():.6f}")
assert torch.allclose(scratch_ce, ref_ce, atol=1e-5), f"CE mismatch: {scratch_ce} vs {ref_ce}"
print("Categorical cross-entropy matches torch ✓")


Categorical CE scratch: 1.627641
F.cross_entropy:        1.627641
Categorical cross-entropy matches torch ✓


## The idiomatic PyTorch way

In practice, use `torch.nn` criterion objects (which wrap `torch.nn.functional`). Always prefer the logits-based variants to avoid manual sigmoid/softmax.

In [9]:
mse_fn = torch.nn.MSELoss()
mae_fn = torch.nn.L1Loss()
bce_logits_fn = torch.nn.BCEWithLogitsLoss()
ce_fn = torch.nn.CrossEntropyLoss()

print("nn.MSELoss:            ", mse_fn(y_hat, y_reg).item())
print("nn.L1Loss:             ", mae_fn(y_hat, y_reg).item())
print("nn.BCEWithLogitsLoss:  ", bce_logits_fn(logits_bin, y_bin).item())
print("nn.CrossEntropyLoss:   ", ce_fn(logits_mc, y_mc).item())

# Sanity-check against scratch
assert torch.allclose(mse_fn(y_hat, y_reg), mse_loss(y_hat, y_reg), atol=1e-6)
assert torch.allclose(mae_fn(y_hat, y_reg), mae_loss(y_hat, y_reg), atol=1e-6)
assert torch.allclose(bce_logits_fn(logits_bin, y_bin), bce_with_logits_loss(logits_bin, y_bin), atol=1e-5)
assert torch.allclose(ce_fn(logits_mc, y_mc), categorical_cross_entropy(logits_mc, y_mc), atol=1e-5)
print("All nn.* losses match from-scratch implementations ✓")


nn.MSELoss:             0.18562085926532745
nn.L1Loss:              0.32583582401275635
nn.BCEWithLogitsLoss:   0.8326990604400635
nn.CrossEntropyLoss:    1.627640962600708
All nn.* losses match from-scratch implementations ✓


## When to use which loss

| Task | Loss | Reason |
|---|---|---|
| Regression, well-behaved targets | MSE | Gaussian likelihood; smooth gradients |
| Regression, heavy-tailed / noisy targets | MAE or Huber | Bounded subgradient; robust to outliers |
| Binary classification | BCE-with-logits | Bernoulli likelihood; numerically stable |
| Multiclass classification | Cross-entropy over logits | Categorical likelihood; log-sum-exp stable |
| Ranking / ordering | Margin / pairwise losses | Optimizes relative separation |

**Key distinction:** the *loss* (MSE, MAE, BCE) defines the training objective and corresponds to a probabilistic likelihood. It is separate from the *regularizer* (L1, L2) that may be added to the loss to constrain weight magnitude.

## Takeaways

- MSE corresponds to Gaussian residuals; large errors receive quadratically large gradients. MAE corresponds to Laplace residuals; its subgradient ±1 is robust but non-smooth.
- BCE is the negative log-likelihood of a Bernoulli distribution. Always use the logits-based form (`BCEWithLogitsLoss` / `binary_cross_entropy_with_logits`) because it avoids computing `sigmoid` explicitly and uses a numerically stable log-sum-exp identity.
- Categorical cross-entropy is the negative log-likelihood of the categorical distribution predicted by softmax. Its gradient with respect to logits is simply `p - y_onehot`.
- The cross-entropy gradient \(p_k - \mathbb{1}[k=y]\) is not 'the gradient of softmax'; it is the joint gradient of softmax composed with negative log-likelihood.
- A loss is the training objective; an evaluation metric measures task success. They are related but not identical — always choose the loss closest to the likelihood model you believe is correct.